In [1]:
import igl
import numpy as np
from src.ms_new import MorseSmale
from src import vis
from src import pathtools
from src import triangletools

import networkx as nx
import pandas as pd

from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import pyvista as pv

from src import shapes

In [3]:
import pickle as pkl

#with open('data/hard-goat.pkl', 'rb') as file:
with open('data/hard-goat-cut.pkl', 'rb') as file:
    vertices, faces, values = pkl.load(file)

print(f'vertices.shape = {vertices.shape}')
print(f'faces.shape = {faces.shape}')
print(f'values.shape = {values.shape}')

vertices.shape = (1022, 3)
faces.shape = (1954, 3)
values.shape = (1022,)


In [4]:
mesh = vis.get_pv_mesh(vertices, faces)

pl = pv.Plotter(window_size=(600, 600))
pl.add_mesh(mesh, scalars=values, cmap="viridis", smooth_shading=False, show_edges=True, opacity=1.0)
pl.show()

Widget(value='<iframe src="http://localhost:46833/index.html?ui=P_0x7ace9e9f1b20_0&reconnect=auto" class="pyvi…

 JS Error => TypeError: Cannot create proxy with a non-object as target or handler
 JS Error => TypeError: Cannot create proxy with a non-object as target or handler


In [5]:
ms = MorseSmale(faces=faces, values=values, vertices=vertices)

In [6]:
(ms.local_maxima, ms.local_minima, ms.saddles)

(array([   0,  148,  596,  906, 1021]),
 array([  55,  315, 1015]),
 array([ 37,  46, 300, 317, 548, 660, 740]))

In [7]:
print(f'local_maxima: {len(ms.local_maxima)}')
print(f'local_minima: {len(ms.local_minima)}')
print(f'saddles: {len(ms.saddles)}')

local_maxima: 5
local_minima: 3
saddles: 7


In [8]:
ms.saddles

array([ 37,  46, 300, 317, 548, 660, 740])

In [9]:
print(f'          ms.local_maxima  = {ms.local_maxima}\nms.values[ms.local_maxima] = {ms.values[ms.local_maxima]}')
print(f'          ms.local_minima  = {ms.local_minima}\nms.values[ms.local_minima] = {ms.values[ms.local_minima]}')
print(f'          ms.saddles  = {ms.saddles}\nms.values[ms.saddles] = {ms.values[ms.saddles]}')


          ms.local_maxima  = [   0  148  596  906 1021]
ms.values[ms.local_maxima] = [ 1.          0.97929624  0.99876966 -0.2022893   0.99996618]
          ms.local_minima  = [  55  315 1015]
ms.values[ms.local_minima] = [-0.98801457 -1.00524038 -1.52792293]
          ms.saddles  = [ 37  46 300 317 548 660 740]
ms.values[ms.saddles] = [-0.9813481   0.59429978 -0.99857391  0.36964455 -0.25429362 -0.94130074
 -0.20907633]


In [10]:
paths = ms.get_paths(how='increasing-decreasing')

Searching paths: 100%|██████████| 24/24 [00:13<00:00,  1.84it/s, Returned directions=set()]


In [11]:
pl = pv.Plotter(window_size=(600, 600))


pl.add_mesh(mesh, scalars=values, cmap="viridis", smooth_shading=False, show_edges=True, opacity=1.0)
pl.add_points(ms.vertices[ms.local_minima], color="lime", point_size=12, render_points_as_spheres=True)
pl.add_points(ms.vertices[ms.local_maxima], color="red", point_size=12, render_points_as_spheres=True)
pl.add_points(ms.vertices[ms.saddles], color="orchid", point_size=12, render_points_as_spheres=True)

for path in ms.get_paths():
    n_sides = len(pathtools.get_path_sides(path, faces))
    path_color = {1: 'orangered', 2: 'white'}[n_sides]

    path_vertices = ms.vertices[path]

    line = pv.lines_from_points(path_vertices)
    pl.add_mesh(
        line,
        color=path_color,
        line_width=6,
        render_lines_as_tubes=True,
    )


pl.show()

Widget(value='<iframe src="http://localhost:46833/index.html?ui=P_0x7ace74fa3a40_1&reconnect=auto" class="pyvi…

In [12]:
pl = pv.Plotter(window_size=(600, 600))

pl.add_mesh(mesh, scalars=values, cmap="viridis", smooth_shading=False, show_edges=True, opacity=1.0)
pl.add_points(ms.vertices[ms.local_minima], color="lime", point_size=12, render_points_as_spheres=True)
pl.add_points(ms.vertices[ms.local_maxima], color="red", point_size=12, render_points_as_spheres=True)
pl.add_points(ms.vertices[ms.saddles], color="orchid", point_size=12, render_points_as_spheres=True)

for path_vertices in tqdm(ms.iterate_paths_close_geodesics(), total=ms.n_paths):
    line = pv.lines_from_points(path_vertices)
    pl.add_mesh(
        line,
        color="white",
        line_width=6,
        render_lines_as_tubes=True,
    )


pl.show()

  0%|          | 0/24 [00:00<?, ?it/s]

Widget(value='<iframe src="http://localhost:46833/index.html?ui=P_0x7ace74fa3ce0_2&reconnect=auto" class="pyvi…

In [13]:
for i in range(len(paths) + 1):
    print(i, len(pathtools.merge_paths_at_nodes([paths[j] for j in range(i)], ms.saddles)))

[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
0 0
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
1 1
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
2 2
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
3 3
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
4 4
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
5 5
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
6 6
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
7 7
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
8 8
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
9 9
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
10 10
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
11 11
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
12 12
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
13 13
[ 37  46 300 317 548 660 740] [[], [], [], [], [], [], []]
14 14
[ 37  46 300 317 548 660 740] [[], [], [], []